In [0]:
from pyspark.sql import functions as f
import sys
sys.path.append('..')
sys.path.append('../..')

import lib_etl.validations_ETL as validations
from lib.job_manager import load_config, split_config

In [0]:
%run ../../config/utils

In [0]:
config = load_config(etl_config_path)
data_paths, club_square_config, config_validation = split_config(config)
run_as_date = dbutils.widgets.get("run_as_date")

### Transform 

In [0]:
df_header = spark.sql(f"""
    SELECT 
        CAST(PURCH_HDR_ID AS LONG) AS PURCH_HDR_ID
        ,CAST(MBRSHP_SID AS LONG) AS MBRSHP_SID
        ,CAST(SITE_NBR AS INT) AS SITE_NBR
        ,CAST(PURCH_DT AS DATE) AS PURCH_DT
        ,SALES_CHANNEL_ID
        ,CAST(TOT_SALES_AMT AS DOUBLE) AS TOT_SALES_AMT
        ,CAST(TAX_AMT AS DOUBLE) AS TAX_AMT
        ,PURCHASE_TM
    FROM 
        {bronze_transaction_header}
""").filter('PURCH_DT >= "2012-09-01"').dropDuplicates()

df_header.createOrReplaceTempView("source")

In [0]:
validations.validate_table(
        spark, "intermediate", 'header', config_validation, df_header, stats_etl_path
    )

### Merge

In [0]:
df_header.write.mode("overwrite").saveAsTable(silver_transaction_header)

if archive_flag:
    save_archive(df_header, silver_transaction_header_archive, run_as_date)